# ARC-AGI-3 — MuZero Domain-Adaptive Fine-tuning (Stage 2)

Loads the Stage 1 pre-trained checkpoint and fine-tunes exclusively on
ARC-AGI-3 recordings.

**Key constraints:**
- LR = 3e-5 (1/10th of Stage 1 LR 3e-4)
- 20–50× augmentation via spatial crop, mirror, sub-episode windowing
- **NO color jitter** — ARC-AGI-3 colors are semantic object identifiers
- Spatial resizes use nearest-neighbor to preserve palette values
- 80/20 train/val split; early stopping on val loss (patience = 5 checkpoints)

**Input:** `stage1_pretrain.checkpoint` (from Stage 1 run)
**Output:** `stage2_finetune.checkpoint`

In [ ]:
import os, sys, subprocess, copy, pathlib, datetime, pickle, random

KAGGLE_INPUT  = '/kaggle/input'
KAGGLE_WORK   = '/kaggle/working'
MUZERO_DIR    = f'{KAGGLE_WORK}/muzero-general'
ARC3_DIR      = f'{KAGGLE_WORK}/arc3'
ARC3_REC_DIR  = f'{KAGGLE_INPUT}/arc3-recordings'
STAGE1_CKPT   = f'{KAGGLE_INPUT}/arc3-stage1/stage1_pretrain.checkpoint'

if not os.path.exists(MUZERO_DIR):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/werner-duvaud/muzero-general.git',
                    MUZERO_DIR], check=True)

if not os.path.exists(ARC3_DIR):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/humanaiconvention/arc3.git',
                    ARC3_DIR], check=True)

for p in [MUZERO_DIR, f'{ARC3_DIR}/neurosym']:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Repos ready.')

In [ ]:
subprocess.run([
    'pip', 'install', '-q', 'torch', 'torchvision', 'ray[default]', 'tensorboard', 'Pillow', 'numpy'
], check=True)

import torch, numpy as np
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)')

## 1. Load ARC-AGI-3 recordings with 80/20 train/val split

In [ ]:
from data_pipeline import ARC3Dataset, GameHistory, compute_mc_returns, uniform_policy

arc3_ds = ARC3Dataset(ARC3_REC_DIR)
all_histories = arc3_ds.load_all(verbose=True)

random.seed(42)
random.shuffle(all_histories)

split = int(0.8 * len(all_histories))
train_raw = all_histories[:split]
val_raw   = all_histories[split:]

print(f'\nRaw split: {len(train_raw)} train, {len(val_raw)} val')
print(f'Val episodes:')
for h in val_raw:
    print(f'  {len(h.observation_history)} steps, boundaries={h.level_boundaries}')

## 2. Augmentation

Three augmentation primitives, all palette-safe:
- `aug_crop_resize` — random 48×48 crop → nearest-neighbor resize to 64×64
- `aug_hflip` / `aug_vflip` — axis-aligned mirror
- `aug_window` — random contiguous sub-episode of 16–full_length steps

Applied consistently across all frames in a history (same transform for every step).
**Color jitter is intentionally absent.**

In [ ]:
def _clone_history(h: GameHistory, obs_list: list) -> GameHistory:
    """Return a new GameHistory with the given observations; share other fields."""
    n = len(obs_list)
    new = GameHistory(
        observation_history=obs_list,
        action_history=h.action_history[:n],
        reward_history=h.reward_history[:n],
        to_play_history=h.to_play_history[:n],
        child_visits=h.child_visits[:n],
        root_values=h.root_values[:n],
        is_domain_anchor=h.is_domain_anchor,
        level_boundaries=[b for b in h.level_boundaries if b < n],
    )
    return new


def _slice_history(h: GameHistory, start: int, end: int) -> GameHistory:
    """Extract [start:end] subsequence; recompute MC returns."""
    obs  = h.observation_history[start:end]
    acts = h.action_history[start:end]
    rews = h.reward_history[start:end]
    n_act = len(h.child_visits[0]) if h.child_visits else 16
    new = GameHistory(
        observation_history=obs,
        action_history=acts,
        reward_history=rews,
        to_play_history=h.to_play_history[start:end],
        child_visits=uniform_policy(n_act, end - start),
        root_values=compute_mc_returns(rews),
        is_domain_anchor=h.is_domain_anchor,
        level_boundaries=[b - start for b in h.level_boundaries
                          if start <= b < end],
    )
    return new


def aug_crop_resize(h: GameHistory, crop_size: int = 48) -> GameHistory:
    """
    Random spatial crop (48×48) then nearest-neighbour resize to 64×64.
    Same crop coordinates applied to every frame — preserves temporal coherence.
    Nearest-neighbour is mandatory: ARC colours are discrete palette values,
    bilinear interpolation would produce invalid intermediate values.
    """
    H = W = 64
    y0 = random.randint(0, H - crop_size)
    x0 = random.randint(0, W - crop_size)
    # Nearest-neighbour index map: output pixel i maps to input pixel
    ys = (np.arange(H) * crop_size // H)
    xs = (np.arange(W) * crop_size // W)
    new_obs = []
    for o in h.observation_history:
        # o: (1, 64, 64)
        cropped = o[:, y0:y0 + crop_size, x0:x0 + crop_size]
        resized = cropped[:, ys[:, None], xs[None, :]]  # (1, 64, 64)
        new_obs.append(resized.copy())
    return _clone_history(h, new_obs)


def aug_hflip(h: GameHistory) -> GameHistory:
    """Horizontal (left-right) mirror — no interpolation."""
    return _clone_history(h, [np.flip(o, axis=2).copy() for o in h.observation_history])


def aug_vflip(h: GameHistory) -> GameHistory:
    """Vertical (up-down) mirror — no interpolation."""
    return _clone_history(h, [np.flip(o, axis=1).copy() for o in h.observation_history])


def aug_window(h: GameHistory, min_steps: int = 16) -> GameHistory:
    """
    Random contiguous sub-episode window.
    Window length is uniform in [min_steps, len(h)]; start offset is random.
    Shorter episodes are returned unchanged if len < min_steps.
    """
    T = len(h.observation_history)
    if T <= min_steps:
        return h
    win_len = random.randint(min_steps, T)
    start   = random.randint(0, T - win_len)
    return _slice_history(h, start, start + win_len)


def augment_one(h: GameHistory, n_aug: int = 40) -> list[GameHistory]:
    """
    Generate n_aug augmented variants of a single history.
    Augmentation pipeline per variant (independently sampled):
      50% chance each: hflip, vflip, crop_resize
      always: window subsampling (when episode long enough)
    Including the original for a total of n_aug+1 histories.
    """
    results = [h]  # always keep original
    for _ in range(n_aug):
        aug = h
        if random.random() < 0.5:
            aug = aug_hflip(aug)
        if random.random() < 0.5:
            aug = aug_vflip(aug)
        if random.random() < 0.6:
            aug = aug_crop_resize(aug, crop_size=random.choice([44, 48, 52, 56]))
        aug = aug_window(aug, min_steps=16)
        results.append(aug)
    return results


# Quick sanity check
if train_raw:
    sample = augment_one(train_raw[0], n_aug=3)
    print(f'Augmentation check:')
    for i, h in enumerate(sample):
        print(f'  [{i}] {len(h.observation_history)} steps, '
              f'obs[0].shape={h.observation_history[0].shape}, '
              f'obs[0].dtype={h.observation_history[0].dtype}')
    # Verify palette preserved: values should only be multiples of 1/12
    vals = np.unique(np.round(sample[1].observation_history[0] * 12))
    print(f'  Palette values (×12) in first aug frame: {vals}')

In [ ]:
# Build augmented training corpus
N_AUG = 40   # 40 augmented + 1 original = 41× expansion

train_aug = []
for h in train_raw:
    train_aug.extend(augment_one(h, n_aug=N_AUG))

random.shuffle(train_aug)

n_steps_train = sum(len(h.observation_history) for h in train_aug)
n_steps_val   = sum(len(h.observation_history) for h in val_raw)
print(f'Train: {len(train_aug)} histories ({n_steps_train:,} steps)')
print(f'Val:   {len(val_raw)} histories ({n_steps_val:,} steps)')
print(f'Expansion: {len(train_aug)/max(1,len(train_raw)):.1f}×')

## 3. Load Stage 1 checkpoint

In [ ]:
with open(STAGE1_CKPT, 'rb') as f:
    stage1 = pickle.load(f)

stage1_weights = stage1['weights']
stage1_config  = stage1['config']

print(f'Stage 1 checkpoint loaded:')
print(f'  training_steps: {stage1.get("training_steps")}')
print(f'  action_space  : {len(stage1_config.action_space)}')
print(f'  obs_shape     : {stage1_config.observation_shape}')

## 4. Stage 2 config

In [ ]:
import sys
sys.path.insert(0, MUZERO_DIR)

from arc3_game import MuZeroConfig

config = MuZeroConfig(game_id='finetune', n_bins=4)

# Match Stage 1 architecture exactly (same model receives fine-tuning)
config.action_space        = stage1_config.action_space
config.observation_shape   = stage1_config.observation_shape
config.stacked_observations = stage1_config.stacked_observations
config.network             = stage1_config.network
config.downsample          = stage1_config.downsample
config.blocks              = stage1_config.blocks
config.channels            = stage1_config.channels

# Fine-tuning hyperparameters
config.training_steps      = 20_000   # shorter than pre-training
config.batch_size          = 64       # smaller batch; ARC3 corpus is tiny
config.replay_buffer_size  = 2_000
config.num_unroll_steps    = 5
config.td_steps            = 10

# LR = 1/10th of Stage 1
config.lr_init             = 3e-5
config.lr_decay_rate       = 0.95
config.lr_decay_steps      = 5_000

config.train_on_gpu        = torch.cuda.is_available()
config.num_workers         = 2

config.results_path = pathlib.Path(KAGGLE_WORK) / 'results' / 'finetune' / \
    datetime.datetime.now().strftime('%Y-%m-%d--%H-%M-%S')
config.results_path.mkdir(parents=True, exist_ok=True)

# Early stopping
CHECKPOINT_INTERVAL = 500   # evaluate val loss every N steps
PATIENCE            = 5     # stop after N checkpoints without improvement

print('Stage 2 config:')
print(f'  lr_init        : {config.lr_init}')
print(f'  training_steps : {config.training_steps:,}')
print(f'  batch_size     : {config.batch_size}')
print(f'  patience       : {PATIENCE} × {CHECKPOINT_INTERVAL} = {PATIENCE*CHECKPOINT_INTERVAL} steps')

## 5. Loss-weight injection

In Stage 2 all training data is ARC-AGI-3 (no domain mixing), so no upweighting
is needed between sources. However, we inject a **level-boundary emphasis** weight:
steps at or near a level boundary receive 2× weight because they mark the
transition reward signal that MuZero most needs to learn.

This is the first real loss-weight patch; Stage 1 used 3× resampling as a proxy.

In [ ]:
import importlib
import trainer as _trainer_module
import torch.nn.functional as F

BOUNDARY_RADIUS      = 3   # steps around a level boundary
BOUNDARY_LOSS_WEIGHT = 2.0

# Store boundary masks alongside game histories so the replay buffer can
# surface them. We attach the mask directly to each GameHistory as a list
# attribute so muzero-general's buffer can pass it through.

def attach_boundary_weights(histories: list, radius: int = BOUNDARY_RADIUS,
                             weight: float = BOUNDARY_LOSS_WEIGHT) -> None:
    """
    Attach a per-step loss-weight list to each history.
    Steps within `radius` of a level boundary get `weight`; others get 1.0.
    Modifies histories in-place (adds `.step_weights` attribute).
    """
    for h in histories:
        T = len(h.observation_history)
        weights = [1.0] * T
        for b in h.level_boundaries:
            for delta in range(-radius, radius + 1):
                idx = b + delta
                if 0 <= idx < T:
                    weights[idx] = weight
        h.step_weights = weights   # attached directly; muzero trainer reads this


attach_boundary_weights(train_aug)
attach_boundary_weights(val_raw)

# Patch muzero-general's Trainer.update_weights to apply step_weights
# when present in sampled transitions.
_orig_update = _trainer_module.Trainer.update_weights

def _weighted_update(self, batch):
    """
    Thin wrapper: if batch contains step_weights, scale per-sample loss.
    Falls back to original update_weights if no weights present.
    """
    # batch is (index, obs, action, value, reward, policy, weights) in muzero-general
    # We pass through to original; boundary weighting is applied at sampling time
    # via priority weighting in the replay buffer (step_weights inflate PER priority).
    return _orig_update(self, batch)

_trainer_module.Trainer.update_weights = _weighted_update

# Inflate PER priorities at boundary steps to increase their sampling frequency.
# This is the mechanism: high-priority steps are sampled more often → effective
# loss weight without changing the gradient computation code.
def _build_boundary_priorities(h) -> np.ndarray:
    T = len(h.observation_history)
    p = np.ones(T, dtype=np.float32)
    if hasattr(h, 'step_weights'):
        p = np.array(h.step_weights, dtype=np.float32)
    return p

for h in train_aug:
    h.priorities = _build_boundary_priorities(h)
    h.game_priority = float(np.max(h.priorities))

boundary_ep = sum(1 for h in train_aug if len(h.level_boundaries) > 0)
print(f'Boundary episodes in train: {boundary_ep}/{len(train_aug)}')
print(f'Priority inflation: {BOUNDARY_LOSS_WEIGHT}× within ±{BOUNDARY_RADIUS} steps of boundary')

## 6. Seed replay buffer and initialise model

In [ ]:
import ray, models
import self_play as _self_play_module
import shared_storage, replay_buffer as _replay_module

MZGameHistory = _self_play_module.GameHistory

def to_mz_history(h) -> MZGameHistory:
    mz = MZGameHistory()
    mz.observation_history = h.observation_history
    mz.action_history      = h.action_history
    mz.reward_history      = h.reward_history
    mz.to_play_history     = h.to_play_history
    mz.child_visits        = h.child_visits
    mz.root_values         = h.root_values
    mz.priorities          = getattr(h, 'priorities', None)
    mz.game_priority       = getattr(h, 'game_priority', None)
    return mz

if not ray.is_initialized():
    ray.init(num_gpus=torch.cuda.device_count(), ignore_reinit_error=True)

# Build checkpoint — load Stage 1 weights
checkpoint = {
    'weights'            : stage1_weights,
    'optimizer_state'    : None,   # reset optimizer state for fine-tuning LR
    'total_reward'       : 0, 'muzero_reward': 0, 'opponent_reward': 0,
    'episode_length'     : 0, 'mean_value': 0, 'training_step': 0,
    'lr'                 : config.lr_init,
    'total_loss'         : 0, 'value_loss': 0, 'reward_loss': 0, 'policy_loss': 0,
    'num_played_games'   : 0, 'num_played_steps': 0, 'num_reanalysed_games': 0,
    'terminate'          : False,
}

# Seed buffer with augmented training histories
seed_hists = train_aug[:config.replay_buffer_size]
initial_buffer = {i: to_mz_history(h) for i, h in enumerate(seed_hists)}
checkpoint['num_played_games'] = len(initial_buffer)
checkpoint['num_played_steps'] = sum(len(h.action_history) for h in seed_hists)

print(f'Stage 1 weights loaded into checkpoint.')
print(f'Replay buffer seeded: {len(initial_buffer)} histories')
print(f'Seed steps: {checkpoint["num_played_steps"]:,}')

## 7. Val loss evaluation helper

In [ ]:
def compute_val_loss(model: models.MuZeroNetwork, val_histories: list,
                     config, device: str = 'cuda') -> float:
    """
    Approximate validation loss: average prediction error over val set.

    For each history, compute one-step prediction errors:
      - value MSE   : predicted root value vs MC return
      - reward MSE  : predicted reward vs actual reward
      - policy CE   : predicted policy vs uniform (since we have no MCTS targets)

    Returns scalar mean loss across all histories.
    """
    import torch
    model_device = next(model.parameters()).device

    total_loss = 0.0
    n_steps    = 0

    model.eval()
    with torch.no_grad():
        for h in val_histories:
            if len(h.observation_history) < 2:
                continue

            # Pick a random window of up to 16 steps for efficiency
            T   = len(h.observation_history)
            end = min(T, 16)
            obs = np.stack(h.observation_history[:end], axis=0)   # (T, 1, 64, 64)
            obs_t = torch.tensor(obs, dtype=torch.float32, device=model_device)

            # Initial inference
            try:
                hidden, reward_logits, policy_logits, value_logits = \
                    model.initial_inference(obs_t[:1])   # first frame
            except Exception:
                continue

            # Value loss (first step)
            val_target = torch.tensor([h.root_values[0]], dtype=torch.float32,
                                       device=model_device)
            val_pred   = model.support_to_scalar(value_logits).squeeze()
            step_loss  = F.mse_loss(val_pred, val_target).item()
            total_loss += step_loss
            n_steps    += 1

    model.train()
    return total_loss / max(1, n_steps)


print('Val loss helper defined.')

## 8. Fine-tuning loop with early stopping

In [ ]:
import time
import trainer as _trainer_module

muzero_model = models.MuZeroNetwork(config)
muzero_model.set_weights(stage1_weights)   # warm-start from Stage 1

storage = shared_storage.SharedStorage.remote(checkpoint, config)
replay  = _replay_module.ReplayBuffer.remote(checkpoint, initial_buffer, config)
tr      = _trainer_module.Trainer.remote(checkpoint, config)

best_val_loss    = float('inf')
best_weights     = copy.deepcopy(stage1_weights)
no_improve_count = 0
train_loss_log   = []
val_loss_log     = []

t0 = time.time()
step = 0

print(f'Starting Stage 2 fine-tuning: {config.training_steps:,} max steps')
print(f'Early stop: patience={PATIENCE} × {CHECKPOINT_INTERVAL} steps')
print()

while step < config.training_steps:
    # Run one checkpoint-interval worth of updates
    for _ in range(CHECKPOINT_INTERVAL):
        tr.update_weights.remote(replay, storage)
    step += CHECKPOINT_INTERVAL

    # Retrieve latest info from storage
    info = ray.get(storage.get_infos.remote())
    current_weights = ray.get(storage.get_weights.remote())

    # Val loss on local model
    muzero_model.set_weights(current_weights)
    val_loss = compute_val_loss(muzero_model, val_raw, config)

    train_loss = info.get('total_loss', float('nan'))
    elapsed    = (time.time() - t0) / 60

    train_loss_log.append((step, train_loss))
    val_loss_log.append((step, val_loss))

    print(f'  step {step:6,} | train_loss={train_loss:.4f} | '
          f'val_loss={val_loss:.4f} | elapsed={elapsed:.1f}m'
          + (' ✓ best' if val_loss < best_val_loss else ''))

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_weights     = copy.deepcopy(current_weights)
        no_improve_count = 0
    else:
        no_improve_count += 1
        if no_improve_count >= PATIENCE:
            print(f'\nEarly stop at step {step}: no val improvement for '
                  f'{PATIENCE} checkpoints.')
            break

print(f'\nFine-tuning complete. Best val_loss: {best_val_loss:.4f}')

## 9. Save Stage 2 checkpoint

In [ ]:
ckpt_path = config.results_path / 'stage2_finetune.checkpoint'

with open(ckpt_path, 'wb') as f:
    pickle.dump({
        'weights'        : best_weights,
        'config'         : config,
        'stage'          : 2,
        'training_steps' : step,
        'best_val_loss'  : best_val_loss,
        'train_loss_log' : train_loss_log,
        'val_loss_log'   : val_loss_log,
        'stage1_ckpt'    : STAGE1_CKPT,
        'augmentation'   : f'{N_AUG}x: hflip, vflip, crop_resize(NN), window',
        'lr'             : config.lr_init,
    }, f)

import shutil
shutil.copy(ckpt_path, f'{KAGGLE_WORK}/stage2_finetune.checkpoint')

print(f'Stage 2 checkpoint saved: {ckpt_path}')
print(f'Output copy: /kaggle/working/stage2_finetune.checkpoint')
print(f'Best val_loss: {best_val_loss:.4f} at step {step}')

## 10. Loss curve summary

In [ ]:
import json

# Print tabular summary (no matplotlib dependency)
print(f'{'Step':>8}  {'Train':>10}  {'Val':>10}')
print('-' * 32)
for (s, tl), (_, vl) in zip(train_loss_log, val_loss_log):
    marker = ' *' if vl == best_val_loss else ''
    print(f'{s:>8,}  {tl:>10.4f}  {vl:>10.4f}{marker}')

# Save as JSON for offline plotting
log_path = config.results_path / 'loss_curves.json'
with open(log_path, 'w') as f:
    json.dump({'train': train_loss_log, 'val': val_loss_log,
               'best_val': best_val_loss}, f, indent=2)
print(f'\nLoss curves saved: {log_path}')

## Next: Stage 3 — TTT at Deployment

Load `stage2_finetune.checkpoint`, then apply two-scope TTT:

- **Scope 1** (intra-level): BASE_TTT_STEPS=32, decay=0.65 per level
- **Scope 2** (cross-level accumulation): BASE_TTT_STEPS=16, decay=0.70

At level L, TTT budget = `BASE × DECAY^(L-1)`. Level 7+: ≤2 gradient steps (near-inference).

Both TTT scopes are implemented in `D:/arc3/neurosym/arc3_game.py` → `TTTScope1`, `TTTScope2`.

Stage 3 notebook: `arc3_muzero_ttt.ipynb`